In [1]:
import torch, random, os, math, json
import numpy as np
from torch import nn
from torch.utils.data import DataLoader
from transformers import AutoTokenizer
from datasets import load_dataset
from tqdm import tqdm
from sklearn.metrics.pairwise import cosine_similarity
import torch.nn.functional as F

In [2]:
# -------------------------------
# Small Transformer Encoder Model
# -------------------------------
class SmallEmbeddingModel(nn.Module):
    def __init__(self, vocab_size, d_model=384, nhead=6, num_layers=6,
                 dim_feedforward=1536, max_seq_length=128, dropout=0.1):
        super().__init__()
        self.d_model = d_model
        self.max_seq_length = max_seq_length

        self.token_embedding = nn.Embedding(vocab_size, d_model)
        self.position_embedding = nn.Embedding(max_seq_length, d_model)

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=nhead, dim_feedforward=dim_feedforward,
            dropout=dropout, batch_first=True, activation='gelu'
        )
        self.transformer_encoder = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
        self.layer_norm = nn.LayerNorm(d_model)
        self.dropout = nn.Dropout(dropout)

        self._init_weights()

    def _init_weights(self):
        nn.init.normal_(self.token_embedding.weight, std=0.02)
        nn.init.normal_(self.position_embedding.weight, std=0.02)

    def mean_pooling(self, token_embeddings, attention_mask):
        mask_expanded = attention_mask.unsqueeze(-1).expand(token_embeddings.size()).float()
        sum_embeddings = torch.sum(token_embeddings * mask_expanded, 1)
        sum_mask = torch.clamp(mask_expanded.sum(1), min=1e-9)
        return sum_embeddings / sum_mask

    def forward(self, input_ids, attention_mask):
        batch_size, seq_len = input_ids.size()
        token_embeds = self.token_embedding(input_ids)
        pos_ids = torch.arange(seq_len, device=input_ids.device).unsqueeze(0).expand(batch_size, -1)
        pos_embeds = self.position_embedding(pos_ids)
        embeddings = self.dropout(self.layer_norm(token_embeds + pos_embeds))
        padding_mask = (attention_mask == 0)
        encoded = self.transformer_encoder(embeddings, src_key_padding_mask=padding_mask)
        sentence_embeddings = self.mean_pooling(encoded, attention_mask)
        return F.normalize(sentence_embeddings, p=2, dim=1)

In [3]:
# -------------------------------
# Triplet Loss
# -------------------------------
class TripletLoss(nn.Module):
    def __init__(self, margin=0.5):
        super().__init__()
        self.margin = margin

    def forward(self, anchor, positive, negative):
        pos_sim = F.cosine_similarity(anchor, positive)
        neg_sim = F.cosine_similarity(anchor, negative)
        loss = torch.clamp(self.margin - (pos_sim - neg_sim), min=0.0)
        return loss.mean()


# -------------------------------
# Collate Function
# -------------------------------
def collate_fn(batch, tokenizer, max_length=128):
    queries = [b['query'] for b in batch]
    positives = [b['pos'][0] if isinstance(b['pos'], list) else b['pos'] for b in batch]
    negatives = [b['neg'][0] if isinstance(b['neg'], list) else b['neg'] for b in batch]

    q_enc = tokenizer(queries, padding=True, truncation=True, max_length=max_length, return_tensors='pt')
    p_enc = tokenizer(positives, padding=True, truncation=True, max_length=max_length, return_tensors='pt')
    n_enc = tokenizer(negatives, padding=True, truncation=True, max_length=max_length, return_tensors='pt')

    return {'query': q_enc, 'positive': p_enc, 'negative': n_enc}


In [4]:
# -------------------------------
# Evaluation Metrics
# -------------------------------
def compute_recall_at_k(similarities, ks=[1, 5, 10, 50, 100]):
    recalls = {}
    for k in ks:
        topk = np.argsort(-similarities, axis=1)[:, :k]
        correct = np.any(topk == 0, axis=1)
        recalls[f"recall@{k}"] = correct.mean()
    return recalls

def compute_precision_at_k(similarities, ks=[1, 5, 10]):
    precisions = {}
    for k in ks:
        topk = np.argsort(-similarities, axis=1)[:, :k]
        correct = np.any(topk == 0, axis=1)
        precisions[f"precision@{k}"] = (correct / k).mean()
    return precisions

def compute_ndcg_at_k(similarities, ks=[10, 50, 100]):
    ndcgs = {}
    for k in ks:
        topk = np.argsort(-similarities, axis=1)[:, :k]
        scores = []
        for row in range(len(topk)):
            pos = np.where(topk[row] == 0)[0]
            if len(pos):
                rank = pos[0] + 1
                scores.append(1.0 / np.log2(rank + 1))
            else:
                scores.append(0)
        ndcgs[f"ndcg@{k}"] = np.mean(scores)
    return ndcgs

def compute_mrr(similarities):
    ranks = np.argsort(-similarities, axis=1)
    positive_ranks = np.where(ranks == 0)[1] + 1
    return np.mean(1.0 / positive_ranks)

In [5]:
# -------------------------------
# Evaluation Loop
# -------------------------------
def evaluate_model(model, loader, device):
    model.eval()
    all_sims = []
    with torch.no_grad():
        for batch in tqdm(loader, desc="Evaluating"):
            q_ids, q_mask = batch['query']['input_ids'].to(device), batch['query']['attention_mask'].to(device)
            p_ids, p_mask = batch['positive']['input_ids'].to(device), batch['positive']['attention_mask'].to(device)
            n_ids, n_mask = batch['negative']['input_ids'].to(device), batch['negative']['attention_mask'].to(device)

            q_emb = model(q_ids, q_mask).cpu().numpy()
            p_emb = model(p_ids, p_mask).cpu().numpy()
            n_emb = model(n_ids, n_mask).cpu().numpy()

            for i in range(len(q_emb)):
                candidates = np.vstack([p_emb[i:i+1], n_emb[i:i+1]])
                sims = cosine_similarity(q_emb[i:i+1], candidates)[0]
                all_sims.append(sims)

    all_sims = np.array(all_sims)
    metrics = {
        **compute_recall_at_k(all_sims),
        **compute_precision_at_k(all_sims),
        **compute_ndcg_at_k(all_sims),
        "MRR": compute_mrr(all_sims)
    }
    return metrics


# -------------------------------
# Training Loop
# -------------------------------
def train_model(model, train_loader, val_loader, optimizer, scheduler, criterion, device,
                num_epochs=3, eval_steps=2000, save_path='best_embedding_model.pt'):

    best_recall = 0.0
    global_step = 0

    for epoch in range(num_epochs):
        model.train()
        total_loss = 0.0
        for step, batch in enumerate(tqdm(train_loader, desc=f"Epoch {epoch+1}/{num_epochs}")):
            q_ids, q_mask = batch['query']['input_ids'].to(device), batch['query']['attention_mask'].to(device)
            p_ids, p_mask = batch['positive']['input_ids'].to(device), batch['positive']['attention_mask'].to(device)
            n_ids, n_mask = batch['negative']['input_ids'].to(device), batch['negative']['attention_mask'].to(device)

            q_emb = model(q_ids, q_mask)
            p_emb = model(p_ids, p_mask)
            n_emb = model(n_ids, n_mask)

            loss = criterion(q_emb, p_emb, n_emb)
            optimizer.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            scheduler.step()

            total_loss += loss.item()
            global_step += 1

            if global_step % eval_steps == 0:
                metrics = evaluate_model(model, val_loader, device)
                print("\n=== Eval Step ===")
                for k, v in metrics.items():
                    print(f"{k:12s}: {v:.4f}")

                if metrics["recall@10"] > best_recall:
                    best_recall = metrics["recall@10"]
                    print(f"New best Recall@10: {best_recall:.4f} -> saving model")
                    torch.save(model.state_dict(), save_path)

        print(f"\nEpoch {epoch+1} Avg Loss: {total_loss / len(train_loader):.4f}\n")

In [6]:
# -------------------------------
# Main
# -------------------------------
def main():
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")
    dataset = load_dataset("w95/triplets")

    print(f"\nDataset: {dataset}")
    total = len(dataset["train"])
    train_end, val_end = int(0.8 * total), int(0.9 * total)

    # Subset for quicker experiments
    USE_FULL = False
    if not USE_FULL:
        train_end, val_end = 100_000, 105_000

    train_ds = dataset["train"].select(range(train_end))
    val_ds = dataset["train"].select(range(train_end, val_end))
    test_ds = dataset["train"].select(range(val_end, val_end + 5000))

    print(f"Train={len(train_ds)}, Val={len(val_ds)}, Test={len(test_ds)}")

    model = SmallEmbeddingModel(tokenizer.vocab_size)
    model.to(device)

    train_loader = DataLoader(train_ds, batch_size=64, shuffle=True,collate_fn=lambda x: collate_fn(x, tokenizer),num_workers=2, pin_memory=True)
    val_loader = DataLoader(val_ds, batch_size=64, shuffle=False,collate_fn=lambda x: collate_fn(x, tokenizer),num_workers=2, pin_memory=True)
    test_loader = DataLoader(test_ds, batch_size=64, shuffle=False,collate_fn=lambda x: collate_fn(x, tokenizer),num_workers=2, pin_memory=True)

    criterion = TripletLoss(0.5)
    optimizer = torch.optim.AdamW(model.parameters(), lr=2e-4, weight_decay=0.01)

    total_steps = len(train_loader) * 3
    warmup_steps = int(0.1 * total_steps)
    scheduler = torch.optim.lr_scheduler.LambdaLR(
        optimizer,
        lambda step: step / warmup_steps if step < warmup_steps else max(0.0, (total_steps - step) / (total_steps - warmup_steps))
    )

    train_model(model, train_loader, val_loader, optimizer, scheduler, criterion, device)

    print("\nEvaluating best checkpoint on TEST set...")
    if os.path.exists("best_embedding_model.pt"):
        model.load_state_dict(torch.load("best_embedding_model.pt", map_location=device))
    metrics = evaluate_model(model, test_loader, device)

    print("\n=== Final Test Metrics ===")
    for k, v in metrics.items():
        print(f"{k:12s}: {v:.4f}")

if __name__ == "__main__":
    main()

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

README.md:   0%|          | 0.00/331 [00:00<?, ?B/s]

...


=== Eval Step ===
recall@1    : 0.6890
recall@5    : 1.0000
recall@10   : 1.0000
recall@50   : 1.0000
recall@100  : 1.0000
precision@1 : 0.6890
precision@5 : 0.2000
precision@10: 0.1000
ndcg@10     : 0.8852
ndcg@50     : 0.8852
ndcg@100    : 0.8852
MRR         : 0.8445


Epoch 3/3: 100%|██████████| 1563/1563 [03:32<00:00,  7.35it/s]



Epoch 3 Avg Loss: 0.2782


Evaluating best checkpoint on TEST set...


Evaluating: 100%|██████████| 79/79 [00:05<00:00, 14.58it/s]



=== Final Test Metrics ===
recall@1    : 0.6554
recall@5    : 1.0000
recall@10   : 1.0000
recall@50   : 1.0000
recall@100  : 1.0000
precision@1 : 0.6554
precision@5 : 0.2000
precision@10: 0.1000
ndcg@10     : 0.8728
ndcg@50     : 0.8728
ndcg@100    : 0.8728
MRR         : 0.8277


In [7]:
# Load the best model checkpoint
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")
model = SmallEmbeddingModel(tokenizer.vocab_size)

if os.path.exists("best_embedding_model.pt"):
    model.load_state_dict(torch.load("best_embedding_model.pt", map_location=device))
    model.to(device)
    model.eval()
    print("Model loaded successfully!")
else:
    print("Best model checkpoint not found!")

Model loaded successfully!


In [11]:
sentences = [
    "This is a sentence about a dog.",
    "Here is another sentence about a cat.",
    "The quick brown fox jumps over the lazy dog.",
    "A cat sat on a mat.",
    "Dogs and cats are common pets.",
    "This sentence is unrelated to animals."
]
# Encode the sentences
encoded_sentences = tokenizer(sentences, padding=True, truncation=True, max_length=128, return_tensors='pt')
input_ids = encoded_sentences['input_ids'].to(device)
attention_mask = encoded_sentences['attention_mask'].to(device)

with torch.no_grad():
    sentence_embeddings = model(input_ids, attention_mask).cpu().numpy()

In [13]:
# Perform retrieval for a query sentence
query_sentence = "Tell me about dogs and cats."

# Encode the query sentence
encoded_query = tokenizer(query_sentence, padding=True, truncation=True, max_length=128, return_tensors='pt')
query_input_ids = encoded_query['input_ids'].to(device)
query_attention_mask = encoded_query['attention_mask'].to(device)

with torch.no_grad():
    query_embedding = model(query_input_ids, query_attention_mask).cpu().numpy()

# Compute similarities with all sentences
similarities = cosine_similarity(query_embedding, sentence_embeddings)[0]

# Get the indices of sentences sorted by similarity
sorted_indices = np.argsort(-similarities)

# Print the query and the retrieved sentences with their similarity scores
print(f"\nQuery: '{query_sentence}'")
print("\nRetrieved Sentences (sorted by similarity):")
for i in sorted_indices:
    print(f"  '{sentences[i]}': {similarities[i]:.4f}")


Query: 'Tell me about dogs and cats.'

Retrieved Sentences (sorted by similarity):
  'Dogs and cats are common pets.': 0.7454
  'The quick brown fox jumps over the lazy dog.': 0.2591
  'This is a sentence about a dog.': 0.2479
  'This sentence is unrelated to animals.': 0.2450
  'Here is another sentence about a cat.': 0.2319
  'A cat sat on a mat.': 0.1223


In [14]:
# Compute and print cosine similarities
print("\nCosine Similarities:")
for i in range(len(sentences)):
    for j in range(i + 1, len(sentences)):
        similarity = cosine_similarity(sentence_embeddings[i].reshape(1, -1), sentence_embeddings[j].reshape(1, -1))[0, 0]
        print(f"'{sentences[i]}' vs '{sentences[j]}': {similarity:.4f}")


Cosine Similarities:
'This is a sentence about a dog.' vs 'Here is another sentence about a cat.': 0.8384
'This is a sentence about a dog.' vs 'The quick brown fox jumps over the lazy dog.': 0.7614
'This is a sentence about a dog.' vs 'A cat sat on a mat.': 0.2356
'This is a sentence about a dog.' vs 'Dogs and cats are common pets.': 0.4243
'This is a sentence about a dog.' vs 'This sentence is unrelated to animals.': 0.8143
'Here is another sentence about a cat.' vs 'The quick brown fox jumps over the lazy dog.': 0.8187
'Here is another sentence about a cat.' vs 'A cat sat on a mat.': 0.3266
'Here is another sentence about a cat.' vs 'Dogs and cats are common pets.': 0.4104
'Here is another sentence about a cat.' vs 'This sentence is unrelated to animals.': 0.7676
'The quick brown fox jumps over the lazy dog.' vs 'A cat sat on a mat.': 0.0123
'The quick brown fox jumps over the lazy dog.' vs 'Dogs and cats are common pets.': 0.4715
'The quick brown fox jumps over the lazy dog.' vs 'T

In [16]:
from scipy.stats import spearmanr

# Define pairs of sentences and their human-rated similarity scores
# (This is a placeholder - in a real scenario, you would have a dataset with human ratings)
sentence_pairs = [
    ("This is a sentence about a dog.", "Here is another sentence about a cat.", 4.0), # Example human rating
    ("This is a sentence about a dog.", "Dogs and cats are common pets.", 3.5),
    ("The quick brown fox jumps over the lazy dog.", "A cat sat on a mat.", 1.0),
    ("Dogs and cats are common pets.", "This sentence is unrelated to animals.", 1.5),
]

# Calculate cosine similarity for these pairs using the trained model
model_similarities = []
human_ratings = []

for sentence1, sentence2, human_rating in sentence_pairs:
    encoded_pair = tokenizer([sentence1, sentence2], padding=True, truncation=True, max_length=128, return_tensors='pt')
    input_ids = encoded_pair['input_ids'].to(device)
    attention_mask = encoded_pair['attention_mask'].to(device)

    with torch.no_grad():
        embeddings = model(input_ids, attention_mask).cpu().numpy()

    cosine_sim = cosine_similarity(embeddings[0].reshape(1, -1), embeddings[1].reshape(1, -1))[0, 0]
    model_similarities.append(cosine_sim)
    human_ratings.append(human_rating)

# Compute Spearman correlation
spearman_corr, p_value = spearmanr(human_ratings, model_similarities)

print(f"\nSpearman Correlation between human ratings and model similarity: {spearman_corr:.4f}")
print(f"P-value: {p_value:.4f}")


Spearman Correlation between human ratings and model similarity: 0.8000
P-value: 0.2000


In [18]:
sts_b_dataset = load_dataset("glue", "stsb")
print(sts_b_dataset)

stsb/train-00000-of-00001.parquet:   0%|          | 0.00/502k [00:00<?, ?B/s]

stsb/validation-00000-of-00001.parquet:   0%|          | 0.00/151k [00:00<?, ?B/s]

stsb/test-00000-of-00001.parquet:   0%|          | 0.00/114k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/5749 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/1500 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1379 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['sentence1', 'sentence2', 'label', 'idx'],
        num_rows: 5749
    })
    validation: Dataset({
        features: ['sentence1', 'sentence2', 'label', 'idx'],
        num_rows: 1500
    })
    test: Dataset({
        features: ['sentence1', 'sentence2', 'label', 'idx'],
        num_rows: 1379
    })
})


In [19]:
sentence1_list = sts_b_dataset['validation']['sentence1']
sentence2_list = sts_b_dataset['validation']['sentence2']
human_similarity_scores = sts_b_dataset['validation']['label']

print(f"Number of sentence pairs in validation set: {len(sentence1_list)}")
print(f"First 5 sentence1: {sentence1_list[:5]}")
print(f"First 5 sentence2: {sentence2_list[:5]}")
print(f"First 5 human similarity scores: {human_similarity_scores[:5]}")

Number of sentence pairs in validation set: 1500
First 5 sentence1: ['A man with a hard hat is dancing.', 'A young child is riding a horse.', 'A man is feeding a mouse to a snake.', 'A woman is playing the guitar.', 'A woman is playing the flute.']
First 5 sentence2: ['A man wearing a hard hat is dancing.', 'A child is riding a horse.', 'The man is feeding a mouse to the snake.', 'A man is playing guitar.', 'A man is playing a flute.']
First 5 human similarity scores: [5.0, 4.75, 5.0, 2.4000000953674316, 2.75]


In [21]:
all_sentences = list(sentence1_list) + list(sentence2_list)

encoded_sentences = tokenizer(all_sentences, padding=True, truncation=True, max_length=128, return_tensors='pt')
input_ids = encoded_sentences['input_ids'].to(device)
attention_mask = encoded_sentences['attention_mask'].to(device)

with torch.no_grad():
    all_sentence_embeddings = model(input_ids, attention_mask).cpu().numpy()

sentence1_embeddings = all_sentence_embeddings[:len(sentence1_list)]
sentence2_embeddings = all_sentence_embeddings[len(sentence1_list):]

print(f"Shape of sentence1 embeddings: {sentence1_embeddings.shape}")
print(f"Shape of sentence2 embeddings: {sentence2_embeddings.shape}")

Shape of sentence1 embeddings: (1500, 384)
Shape of sentence2 embeddings: (1500, 384)


In [22]:
calculated_similarities = []
for i in range(len(sentence1_embeddings)):
    similarity = cosine_similarity(sentence1_embeddings[i].reshape(1, -1), sentence2_embeddings[i].reshape(1, -1))[0, 0]
    calculated_similarities.append(similarity)

print(f"First 5 calculated similarity scores: {calculated_similarities[:5]}")

First 5 calculated similarity scores: [np.float32(0.96034706), np.float32(0.87231433), np.float32(0.9604667), np.float32(0.9418205), np.float32(0.9327505)]


In [23]:
from scipy.stats import spearmanr

spearman_corr, p_value = spearmanr(human_similarity_scores, calculated_similarities)

print(f"\nSpearman Correlation between human ratings and model similarity: {spearman_corr:.4f}")
print(f"P-value: {p_value:.4f}")


Spearman Correlation between human ratings and model similarity: 0.3768
P-value: 0.0000


In [24]:
print(f"\nSpearman Correlation between human ratings and model similarity: {spearman_corr:.4f}")
print(f"P-value: {p_value:.4f}")


Spearman Correlation between human ratings and model similarity: 0.3768
P-value: 0.0000


## Summary:

### Data Analysis Key Findings

*   The "glue" dataset with the "stsb" configuration was successfully loaded, containing 'train', 'validation', and 'test' splits, each with 'sentence1', 'sentence2', 'label', and 'idx' features.
*   The validation split contains 1500 sentence pairs.
*   The model successfully generated embeddings for all sentences.
*   The cosine similarity between the encoded sentence pairs was calculated, with initial checks showing plausible scores between 0.87 and 0.96 for the first five pairs.
*   The Spearman correlation coefficient between the human-rated similarity scores and the model-calculated similarity scores is approximately 0.3768.
*   The p-value for the Spearman correlation is approximately 0.0000, indicating statistical significance.

### Insights or Next Steps

*   The moderate positive Spearman correlation of approximately 0.38 suggests that the pre-trained model's cosine similarity scores have a statistically significant but not very strong agreement with human judgments of sentence similarity on the STS-B validation set.
*   Further steps could involve fine-tuning the pre-trained model on the STS-B training data to potentially improve the correlation with human ratings, or exploring other sentence embedding models and similarity metrics.
